In [ ]:
import requests
from bs4 import BeautifulSoup

SITE_STRUCTURE = 'https://www.fisheries.noaa.gov/species-directory?page='
BASE_SITE = 'https://www.fisheries.noaa.gov/'
PAGE_LAST = 12

In [ ]:
def get_table_data(page_data):
    soup = BeautifulSoup(page_data, 'html.parser')
    table_list = soup.find_all('tbody')
    table = None
    if len(table_list) == 0:
        return []
    else:
        table = table_list[0]
        links = table.find_all('a')
        for i in range(len(links)):
            links[i] = links[i].get('href')
        return links

In [ ]:
all_links = []

for page in range(0, PAGE_LAST+1):
    new_page = SITE_STRUCTURE+str(page)
    response = requests.get(new_page)
    all_links.extend(get_table_data(response.text))

In [ ]:
all_links

In [ ]:
from urllib.parse import urljoin

def get_image_link(link):
    url_path = urljoin(BASE_SITE, link)
    response = requests.get(url_path)
    response.raise_for_status()
    page_data = response.text

    soup = BeautifulSoup(page_data, 'html.parser')
    image = soup.find("div", {"class": "species-overview"}).find("img")
    if image is not None:
        return urljoin(BASE_SITE, image.get("src"))
    else:
        return None
    

get_image_link(all_links[222])

In [ ]:
img_links = {
    link: img_link
    for link in all_links
    if (img_link := get_image_link(link)) is not None
}

In [ ]:
img_links

In [ ]:
import json
JSON_PATH = '/kaggle/working/image_site.json'
with open(JSON_PATH, 'w') as f:
    json.dump(img_links, f, indent=4)